# Load saved models


In [ ]:
import joblib

# Load all saved models
from jpeg_compression_detection.paths import (
    DATA_DIR,
    MODELS_DIR,
    FIGURES_DIR,
    display_path,
)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
model_files = list(MODELS_DIR.glob('*_compression_model.pkl'))

loaded_results = {}

print("Loading saved models...")
print("=" * 70)

for model_path in model_files:
    # Extract the model name from the filename
    model_filename = model_path.name
    # Remove '_compression_model.pkl' and replace underscores with spaces
    model_name = model_filename.replace('_compression_model.pkl', '').replace('_', ' ').title()
    
    # Skip the best-model alias
    if model_name == 'Best':
        continue
    
    # Load the model
    model = joblib.load(model_path)
    loaded_results[model_name] = {'model': model}
    
    print(f"✓ Loaded: {model_name}")
    print(f"  File: {model_filename}")
    print(f"  Type: {type(model).__name__}")
    print()

print(f"\nLoaded {len(loaded_results)} models")
print("=" * 70)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Font-size parameter used by all plots
FONT_SIZE = 16

# Load the JFIF data
data_path = DATA_DIR / 'results_jfif.csv'
from_csv = pd.read_csv(data_path)
from_csv = from_csv.dropna()
X_from_csv = from_csv.drop(['filename', 'extension', 'estimated_quality'], axis=1)

print(f"JFIF data loaded: {len(X_from_csv)} samples")
print("=" * 70)

# Predict quality for the JFIF images
y_pred_jfif_loaded = {}
for name, res in loaded_results.items():
    y_pred_jfif_loaded[name] = res['model'].predict(X_from_csv)
    print(f"✓ Prediction completed: {name}")

print("\n" + "=" * 70)

# Visualize predictions from all models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, (name, y_pred) in enumerate(y_pred_jfif_loaded.items()):
    # Histogram of predicted values
    axes[idx].hist(y_pred, bins=50, alpha=0.7, edgecolor='black', color='skyblue')
    axes[idx].axvline(x=y_pred.mean(), color='r', linestyle='--', lw=2, 
                      label=f'Mean = {y_pred.mean():.2f}')
    axes[idx].axvline(x=np.median(y_pred), color='g', linestyle='--', lw=2, 
                      label=f'Median = {np.median(y_pred):.2f}')
    axes[idx].set_xlabel('Predicted image quality', fontsize=FONT_SIZE)
    axes[idx].set_ylabel('Number of images', fontsize=FONT_SIZE)
    axes[idx].set_title(f'{name}\nMin={y_pred.min():.1f}, Max={y_pred.max():.1f}, Std={y_pred.std():.2f}', 
                        fontsize=FONT_SIZE)
    # axes[idx].set_yscale('log')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend(fontsize=FONT_SIZE)
    axes[idx].set_xlim(0, 100)
    axes[idx].tick_params(axis='both', which='major', labelsize=FONT_SIZE)

plt.tight_layout()
# plt.savefig(FIGURES_DIR / 'jfif_quality_predictions_loaded_models.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nQuality-prediction statistics for JFIF images (loaded models):")
print("="*70)
for name, y_pred in y_pred_jfif_loaded.items():
    print(f"\n{name}:")
    print(f"  Mean: {y_pred.mean():.2f}")
    print(f"  Median: {np.median(y_pred):.2f}")
    print(f"  Std. dev.: {y_pred.std():.2f}")
    print(f"  Min: {y_pred.min():.2f}")
    print(f"  Max: {y_pred.max():.2f}")
    print(f"  Q1 (25%): {np.percentile(y_pred, 25):.2f}")
    print(f"  Q3 (75%): {np.percentile(y_pred, 75):.2f}")

print("\n" + "="*70)


# Prediction with one model


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_path = DATA_DIR / 'results_ppm.csv'

# Load the PPM data
from_csv = pd.read_csv(data_path)
from_csv = from_csv.dropna()
X_from_csv = from_csv.drop(['filename', 'extension', 'estimated_quality'], axis=1)

print(f"PPM data loaded: {len(X_from_csv)} samples")
print("=" * 70)

# Select one model (for example, Gradient Boosting)
model_name = 'Random Forest'
model = loaded_results[model_name]['model']

# Predict quality for the PPM images
y_pred = model.predict(X_from_csv)
print(f"✓ Prediction completed: {model_name}")

print("\n" + "=" * 70)

# Visualize the predictions
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(y_pred, bins=50, alpha=0.7, edgecolor='black', color='skyblue')
ax.axvline(x=y_pred.mean(), color='r', linestyle='--', lw=2, 
          label=f'Mean = {y_pred.mean():.2f}')
ax.axvline(x=np.median(y_pred), color='g', linestyle='--', lw=2, 
          label=f'Median = {np.median(y_pred):.2f}')
ax.set_xlabel('Predicted image quality', fontsize=FONT_SIZE)
ax.set_ylabel('Number of images', fontsize=FONT_SIZE)
ax.set_title(f'{model_name}\nMin={y_pred.min():.1f}, Max={y_pred.max():.1f}, Std={y_pred.std():.2f}', 
            fontsize=FONT_SIZE)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=FONT_SIZE)
ax.set_xlim(0, 100)
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)

plt.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure_path = FIGURES_DIR / 'quality_predictions_single_model.png'
plt.savefig(figure_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\nQuality-prediction statistics for PPM images ({model_name}):")
print("="*70)
print(f"  Mean: {y_pred.mean():.2f}")
print(f"  Median: {np.median(y_pred):.2f}")
print(f"  Std. dev.: {y_pred.std():.2f}")
print(f"  Min: {y_pred.min():.2f}")
print(f"  Max: {y_pred.max():.2f}")
print(f"  Q1 (25%): {np.percentile(y_pred, 25):.2f}")
print(f"  Q3 (75%): {np.percentile(y_pred, 75):.2f}")
print("\n" + "="*70)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

FONT_SIZE = 16
evaluation_data_path = DATA_DIR / 'results_humanFacesDatasetAI.csv'

if not evaluation_data_path.exists():
    print(f"Evaluation table is missing: {display_path(evaluation_data_path)}")
else:
    dataset = pd.read_csv(evaluation_data_path).dropna()
    X_optional = dataset.drop(['filename', 'extension', 'estimated_quality'], axis=1)
    print(f"Evaluation dataset loaded: {len(X_optional)} samples")
    print("=" * 70)

    model_name = 'Random Forest'
    model = loaded_results[model_name]['model']
    y_pred = model.predict(X_optional)
    print(f"Prediction completed: {model_name}")

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(y_pred, bins=50, alpha=0.7, edgecolor='black', color='skyblue')
    ax.axvline(x=y_pred.mean(), color='r', linestyle='--', lw=2,
               label=f'Mean = {y_pred.mean():.2f}')
    ax.axvline(x=np.median(y_pred), color='g', linestyle='--', lw=2,
               label=f'Median = {np.median(y_pred):.2f}')
    ax.set_xlabel('Predicted image quality', fontsize=FONT_SIZE)
    ax.set_ylabel('Number of images', fontsize=FONT_SIZE)
    ax.set_title(
        f'{model_name}\nMin={y_pred.min():.1f}, Max={y_pred.max():.1f}, Std={y_pred.std():.2f}',
        fontsize=FONT_SIZE,
    )
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=FONT_SIZE)
    ax.set_xlim(0, 100)
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
    plt.tight_layout()
    plt.show()

    print(f"\nQuality-prediction statistics ({model_name}):")
    print("=" * 70)
    print(f"  Mean: {y_pred.mean():.2f}")
    print(f"  Median: {np.median(y_pred):.2f}")
    print(f"  Std. dev.: {y_pred.std():.2f}")
    print(f"  Min: {y_pred.min():.2f}")
    print(f"  Max: {y_pred.max():.2f}")
    print(f"  Q1 (25%): {np.percentile(y_pred, 25):.2f}")
    print(f"  Q3 (75%): {np.percentile(y_pred, 75):.2f}")


# Overlay multiple datasets


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

FONT_SIZE = 13

def _min_max_normalize_to_0_100(values):
    values = np.asarray(values, dtype=float)
    v_min = np.min(values)
    v_max = np.max(values)
    if np.isclose(v_min, v_max):
        # If all values are equal, return the constant value 50
        return np.full_like(values, 50.0, dtype=float)
    return (values - v_min) / (v_max - v_min) * 100.0

def plot_overlay_histograms_from_csv_list(
    csv_paths,
    dataset_names,
    loaded_results,
    model_name='Random Forest',
    bins=50,
    alpha=0.35,
    xlim=(0, 100),
    normalize_each_dataset=True,
):
    """
    Plot overlaid predicted-quality histograms for multiple CSV files
    and print statistics for every dataset.
    normalize_each_dataset=True applies min-max normalization to each dataset separately.
    """
    if len(csv_paths) != len(dataset_names):
        raise ValueError('csv_paths and dataset_names must have the same length.')

    if model_name not in loaded_results:
        raise ValueError(
            f"Model '{model_name}' is not present in loaded_results. Available: {list(loaded_results.keys())}"
        )

    model = loaded_results[model_name]['model']
    predictions = {}
    stats_rows = []

    # Use one consistent color palette across datasets
    cmap = plt.cm.get_cmap('tab10', len(csv_paths))

    for i, (csv_path, label) in enumerate(zip(csv_paths, dataset_names)):
        csv_path = Path(csv_path)
        if not csv_path.exists():
            print(f"⚠ Skipped (file not found): {display_path(csv_path)}")
            continue

        from_csv = pd.read_csv(csv_path).dropna()

        required_drop_cols = ['filename', 'extension', 'estimated_quality']
        existing_drop_cols = [c for c in required_drop_cols if c in from_csv.columns]
        X_from_csv = from_csv.drop(existing_drop_cols, axis=1)

        if X_from_csv.empty:
            print(f"⚠ Skipped (no features remain after dropping metadata): {display_path(csv_path)}")
            continue

        y_pred_raw = model.predict(X_from_csv)
        y_pred = _min_max_normalize_to_0_100(y_pred_raw) if normalize_each_dataset else y_pred_raw
        predictions[label] = y_pred

        stats_rows.append({
            'dataset': label,
            'n': len(y_pred),
            'mean': float(np.mean(y_pred)),
            'median': float(np.median(y_pred)),
            'std': float(np.std(y_pred)),
            'min': float(np.min(y_pred)),
            'q1_25%': float(np.percentile(y_pred, 25)),
            'q3_75%': float(np.percentile(y_pred, 75)),
            'max': float(np.max(y_pred)),
            'color': cmap(i),
        })

        suffix = ' [normalized per dataset]' if normalize_each_dataset else ''
        print(f"✓ Prediction completed: {label} ({len(y_pred)} samples){suffix}")

    if not predictions:
        raise ValueError("No valid data are available for plotting.")

    fig, ax = plt.subplots(figsize=(14, 8))

    for i, (label, y_pred) in enumerate(predictions.items()):
        color = cmap(i)
        ax.hist(
            y_pred,
            bins=bins,
            alpha=alpha,
            edgecolor='black',
            linewidth=0.6,
            color=color,
            label=f"{label} (μ={np.mean(y_pred):.1f})",
        )

    ax.set_xlabel(
        'Predicted image quality (normalized per dataset to [0, 100])' if normalize_each_dataset else 'Predicted image quality',
        fontsize=FONT_SIZE,
    )
    ax.set_ylabel('Number of images', fontsize=FONT_SIZE)
    ax.set_title(f'Overlaid histograms of predicted quality\nModel: {model_name}', fontsize=FONT_SIZE + 1)
    ax.set_xlim(*xlim)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=FONT_SIZE - 2, ncol=2)
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE - 1)

    # Place the statistics below the plot
    stats_df = pd.DataFrame(stats_rows).drop(columns=['color'])
    stats_df = stats_df.sort_values('dataset').reset_index(drop=True)

    stats_text_lines = ["Prediction statistics:"]
    for _, row in stats_df.iterrows():
        stats_text_lines.append(
            f"{row['dataset']}: n={int(row['n'])}, mean={row['mean']:.2f}, median={row['median']:.2f}, std={row['std']:.2f}, "
            f"min={row['min']:.2f}, q1={row['q1_25%']:.2f}, q3={row['q3_75%']:.2f}, max={row['max']:.2f}"
        )

    plt.subplots_adjust(bottom=0.34)
    fig.text(
        0.01,
        0.01,
        "\n".join(stats_text_lines),
        ha='left',
        va='bottom',
        fontsize=10,
        family='monospace',
    )

    plt.show()

    print("\n" + "=" * 100)
    print("STATISTICS (table)")
    print("=" * 100)
    display(stats_df.round(3))

    return stats_df, predictions

# Example: CSV files whose distributions will be compared
CSV_PATHS = [
    DATA_DIR / 'results_jfif.csv',
    DATA_DIR / 'results_ppm.csv',
    DATA_DIR / 'results_humanFacesDatasetAI.csv',
    DATA_DIR / 'results_humanFacesDatasetReal.csv',
]
DATASET_NAMES = [
    'JFIF',
    'PPM',
    'Human Faces - AI',
    'Human Faces - Real',
]

stats_df, predictions = plot_overlay_histograms_from_csv_list(
    csv_paths=CSV_PATHS,
    dataset_names=DATASET_NAMES,
    loaded_results=loaded_results,
    model_name='Random Forest',
    bins=55,
    alpha=0.35,
    xlim=(0, 100),
    normalize_each_dataset=True,
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

FONT_SIZE = 13


def _min_max_normalize_to_0_100(values):
    values = np.asarray(values, dtype=float)
    v_min = np.min(values)
    v_max = np.max(values)
    if np.isclose(v_min, v_max):
        return np.full_like(values, 50.0, dtype=float)
    return (values - v_min) / (v_max - v_min) * 100.0


def plot_line_envelopes_from_csv_list(
    csv_paths,
    dataset_names,
    loaded_results,
    model_name='Random Forest',
    bins=55,
    xlim=(0, 100),
    normalize_each_dataset=True,
    density=False,
):
    """
    Plot predicted-quality histogram envelopes as lines for multiple CSV files.
    normalize_each_dataset=True applies min-max normalization to each dataset separately.
    density=True uses normalized density on the y-axis; False uses image counts.
    """
    if len(csv_paths) != len(dataset_names):
        raise ValueError('csv_paths and dataset_names must have the same length.')

    if model_name not in loaded_results:
        raise ValueError(
            f"Model '{model_name}' is not present in loaded_results. "
            f"Available: {list(loaded_results.keys())}"
        )

    model = loaded_results[model_name]['model']
    cmap = plt.cm.get_cmap('tab10', len(csv_paths))
    stats_rows = []
    plot_data = []

    for i, (csv_path, label) in enumerate(zip(csv_paths, dataset_names)):
        csv_path = Path(csv_path)
        if not csv_path.exists():
            print(f"⚠ Skipped (file not found): {display_path(csv_path)}")
            continue

        from_csv = pd.read_csv(csv_path).dropna()
        required_drop_cols = ['filename', 'extension', 'estimated_quality']
        existing_drop_cols = [c for c in required_drop_cols if c in from_csv.columns]
        X_from_csv = from_csv.drop(existing_drop_cols, axis=1)

        if X_from_csv.empty:
            print(f"⚠ Skipped (no features remain after dropping metadata): {display_path(csv_path)}")
            continue

        y_pred_raw = model.predict(X_from_csv)
        y_pred = _min_max_normalize_to_0_100(y_pred_raw) if normalize_each_dataset else y_pred_raw
        color = cmap(i)

        counts, bin_edges = np.histogram(y_pred, bins=bins, range=xlim, density=density)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        plot_data.append((label, bin_centers, counts, color, y_pred))

        stats_rows.append({
            'dataset': label,
            'n': len(y_pred),
            'mean': float(np.mean(y_pred)),
            'median': float(np.median(y_pred)),
            'std': float(np.std(y_pred)),
            'min': float(np.min(y_pred)),
            'q1_25%': float(np.percentile(y_pred, 25)),
            'q3_75%': float(np.percentile(y_pred, 75)),
            'max': float(np.max(y_pred)),
        })

        suffix = ' [normalized per dataset]' if normalize_each_dataset else ''
        print(f"✓ Prediction completed: {label} ({len(y_pred)} samples){suffix}")

    if not plot_data:
        raise ValueError("No valid data are available for plotting.")

    fig, ax = plt.subplots(figsize=(14, 8))

    for label, bin_centers, counts, color, y_pred in plot_data:
        ax.plot(
            bin_centers, counts,
            color=color, lw=2,
            label=f"{label} (μ={np.mean(y_pred):.1f})",
        )
        ax.fill_between(bin_centers, counts, alpha=0.08, color=color)

    ylabel = 'Density' if density else 'Number of images'
    xlabel = (
        'Predicted image quality (normalized per dataset to [0, 100])'
        if normalize_each_dataset else 'Predicted image quality'
    )
    ax.set_xlabel(xlabel, fontsize=FONT_SIZE)
    ax.set_ylabel(ylabel, fontsize=FONT_SIZE)
    ax.set_title(
        f'Predicted quality distribution envelopes\nModel: {model_name}',
        fontsize=FONT_SIZE + 1,
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=FONT_SIZE - 2, ncol=2)
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE - 1)

    stats_df = pd.DataFrame(stats_rows).sort_values('dataset').reset_index(drop=True)

    stats_text_lines = ["Prediction statistics:"]
    for _, row in stats_df.iterrows():
        stats_text_lines.append(
            f"{row['dataset']}: n={int(row['n'])}, mean={row['mean']:.2f}, "
            f"median={row['median']:.2f}, std={row['std']:.2f}, "
            f"min={row['min']:.2f}, q1={row['q1_25%']:.2f}, "
            f"q3={row['q3_75%']:.2f}, max={row['max']:.2f}"
        )

    n_text_lines = len(stats_text_lines)
    bottom_margin = max(0.20, 0.04 * n_text_lines)
    plt.subplots_adjust(bottom=bottom_margin)
    fig.text(
        0.01, 0.01,
        "\n".join(stats_text_lines),
        ha='left', va='bottom',
        fontsize=10, family='monospace',
    )

    plt.show()

    print("\n" + "=" * 100)
    print("STATISTICS (table)")
    print("=" * 100)
    display(stats_df.round(3))

    return stats_df


CSV_PATHS = [
    DATA_DIR / 'results_jfif.csv',
    DATA_DIR / 'results_ppm.csv',
    DATA_DIR / 'results_humanFacesDatasetAI.csv',
    DATA_DIR / 'results_humanFacesDatasetReal.csv',
]
DATASET_NAMES = [
    'JFIF',
    'PPM',
    'Human Faces - AI',
    'Human Faces - Real',
]

plot_line_envelopes_from_csv_list(
    csv_paths=CSV_PATHS,
    dataset_names=DATASET_NAMES,
    loaded_results=loaded_results,
    model_name='Random Forest',
    bins=55,
    xlim=(0, 100),
    normalize_each_dataset=True,
    density=False,
)


# Single-image live prediction


In [28]:
from jpeg_compression_detection import detect_jpeg_compression

def check_image(image_path):
    """Extract indicators and predict JPEG quality for one image."""
    if not image_path.exists():
        print(f"Skipping optional example because it is not available: {display_path(image_path)}")
        return

    print(f"Analyzing file: {display_path(image_path)}")
    result = detect_jpeg_compression(image_path)

    if not isinstance(result, dict):
        print("The image indicators could not be computed.")
        return

    loaded_feature_columns = [
        'blockiness_0', 'blockiness_1', 'blockiness_2', 'blockiness_3',
        'blockiness_4', 'blockiness_5', 'blockiness_6', 'blockiness_7',
        'blockiness_max', 'blockiness_mean', 'blockiness_std',
        'energy_mean', 'energy_std', 'psnr',
    ]
    feature_dict = {column: result.get(column, 0) for column in loaded_feature_columns}
    X_single = pd.DataFrame([feature_dict])

    if X_single.isnull().any().any():
        print("\nMissing values in the indicators:")
        print(X_single.isnull().sum()[X_single.isnull().sum() > 0])
        return

    model_name = 'Random Forest'
    model = loaded_results[model_name]['model']
    y_pred = model.predict(X_single)

    print(f"\nPredicted image quality ({model_name}): {y_pred[0]:.2f}")
    print("-" * 80)


In [ ]:
example_paths = [
    DATA_DIR / 'examples' / 'fakeface.jpg',
    DATA_DIR / 'examples' / 'fakeface-100x100.png',
    DATA_DIR / 'examples' / 'fakeface-500x500.png',
    DATA_DIR / 'examples' / 'fakeface-250x250.png',
    DATA_DIR / 'examples' / 'test-500-700q60q100-1000x1500.png',
]

for example_path in example_paths:
    check_image(example_path)
